In [2]:
import numpy as np

# Data: draw N samples from a D-dimensional Gaussian
np.random.seed(0)

N, D = 500, 2
mu_true = np.array([0.0, 0.0])
Sigma_true = np.array([[4.0, 2.0],
                       [2.0, 16.0]])

X = np.random.multivariate_normal(mu_true, Sigma_true, size=N)  # shape (N, D)

# Batch MLE (Gaussian):
#   mu_ML = (1/N) Σ x_n
#   Σ_ML  = (1/N) Σ (x_n - mu_ML)(x_n - mu_ML)^T
# Note: Σ_ML is biased; unbiased sample covariance uses 1/(N-1).
mu_ML = X.mean(axis=0)

Xc = X - mu_ML
Sigma_ML = (Xc.T @ Xc) / N          # MLE covariance (biased)
Sigma_unbiased = (Xc.T @ Xc) / (N-1) # unbiased covariance estimate

print("Batch estimates")
print("mu_ML =", mu_ML)
print("Sigma_ML (biased, 1/N) =\n", Sigma_ML)
print("Sigma_unbiased (1/(N-1)) =\n", Sigma_unbiased)

# Sequential estimator (Bishop Sec. 2.3.5 style):
# Let m_n be mean after n points.
# Let S_n = Σ_{i=1..n} (x_i - m_n)(x_i - m_n)^T   (scatter matrix)
# Recursion:
#   m_n = m_{n-1} + (x_n - m_{n-1})/n
#   S_n = S_{n-1} + (x_n - m_{n-1})(x_n - m_n)^T
# Then:
#   Σ_ML = S_N / N,   Σ_unbiased = S_N / (N-1)
m = np.zeros(D)
S = np.zeros((D, D))

for n, x in enumerate(X, start=1):
    delta = x - m
    m_new = m + delta / n
    S += np.outer(delta, (x - m_new))  # same as outer(delta, delta_new)
    m = m_new

mu_seq = m
Sigma_ML_seq = S / N
Sigma_unbiased_seq = S / (N - 1)

print("\nSequential estimates")
print("mu_seq =", mu_seq)
print("Sigma_ML_seq (biased, 1/N) =\n", Sigma_ML_seq)
print("Sigma_unbiased_seq (1/(N-1)) =\n", Sigma_unbiased_seq)


Batch estimates
mu_ML = [-0.08989364 -0.25296839]
Sigma_ML (biased, 1/N) =
 [[ 3.9526493   1.79028998]
 [ 1.79028998 15.19459535]]
Sigma_unbiased (1/(N-1)) =
 [[ 3.96057044  1.79387774]
 [ 1.79387774 15.22504544]]

Sequential estimates
mu_seq = [-0.08989364 -0.25296839]
Sigma_ML_seq (biased, 1/N) =
 [[ 3.9526493   1.79028998]
 [ 1.79028998 15.19459535]]
Sigma_unbiased_seq (1/(N-1)) =
 [[ 3.96057044  1.79387774]
 [ 1.79387774 15.22504544]]


In this exercise we estimate the parameters of a multivariate Gaussian from i.i.d. samples
$\{x_n\}_{n=1}^N$. The maximum-likelihood (ML) estimators (Bishop, Ch. 2) are

$$
\hat{\mu}_{\mathrm{ML}}=\frac{1}{N}\sum_{n=1}^N x_n,\qquad
\hat{\Sigma}_{\mathrm{ML}}=\frac{1}{N}\sum_{n=1}^N (x_n-\hat{\mu}_{\mathrm{ML}})(x_n-\hat{\mu}_{\mathrm{ML}})^{\mathsf T}.
$$

The sequential (online) estimator in Section 2.3.5 updates the sample mean and the scatter
matrix after each new observation, producing exactly the same final ML estimates as the batch
formulas, but without storing all data.

Bias enters through the covariance estimate. The mean estimator is unbiased:

$$
\mathbb{E}\!\left[\hat{\mu}_{\mathrm{ML}}\right]=\mu.
$$

But the ML covariance is biased downward for finite $N$ because it uses $\hat{\mu}$ computed from
the same data, reducing the effective degrees of freedom:

$$
\mathbb{E}\!\left[\hat{\Sigma}_{\mathrm{ML}}\right]=\frac{N-1}{N}\,\Sigma.
$$

A standard remedy is the unbiased (Bessel-corrected) covariance estimator:

$$
\hat{\Sigma}_{\mathrm{unb}}=\frac{1}{N-1}\sum_{n=1}^N (x_n-\hat{\mu})(x_n-\hat{\mu})^{\mathsf T}
=\frac{N}{N-1}\,\hat{\Sigma}_{\mathrm{ML}}.
$$
In practice, the bias shrinks as $N$ grows (consistency), but for small samples, the correction is
important when we want an unbiased estimate of population covariance.
